In [ ]:
!pip install sentence-transformers faiss-cpu scikit-fuzzy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 28.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle, time
import numpy as np
import faiss
import skfuzzy as fuzz
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

BASE = '/content/drive/MyDrive/semantic-search-system'

embeddings = np.load(f'{BASE}/models/embeddings.npy')
index      = faiss.read_index(f'{BASE}/models/faiss.index')

with open(f'{BASE}/models/cluster_model.pkl', 'rb') as f:
    cluster_model = pickle.load(f)

membership_matrix = np.load(f'{BASE}/models/membership_matrix.npy')

with open(f'{BASE}/data/processed/clean_corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

model = SentenceTransformer('BAAI/bge-base-en-v1.5')

print(f'Corpus  : {len(corpus["texts"])} docs')
print(f'FAISS   : {index.ntotal} vectors')
print(f'Clusters: {cluster_model["n_clusters"]}')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Corpus  : 13574 docs
FAISS   : 13574 vectors
Clusters: 15


In [ ]:
@dataclass
class CacheEntry:
    query: str; query_embedding: np.ndarray
    result: Any; cluster_id: int
    timestamp: float = field(default_factory=time.time)
    hit_count: int = 0

class SemanticCache:
    def __init__(self, thr=0.85):
        self.thr=thr; self._c={}; self._lu=self._hi=0

    def lookup(self, emb, cid):
        self._lu += 1
        bs, be = -1.0, None
        for e in self._c.get(cid, []):
            a = emb.flatten().astype(np.float64)
            b = e.query_embedding.flatten().astype(np.float64)
            d = np.linalg.norm(a) * np.linalg.norm(b)
            s = float(np.dot(a,b)/d) if d>1e-9 else 0.0
            if s > bs: bs, be = s, e
        if be and bs >= self.thr:
            self._hi += 1; be.hit_count += 1
            return {'hit':True,'matched_query':be.query,
                    'similarity_score':float(bs),'result':be.result,'cluster_id':cid}
        return None

    def store(self, q, emb, res, cid):
        self._c.setdefault(cid, []).append(
            CacheEntry(q, emb.copy(), res, cid))

    def stats(self):
        total = sum(len(v) for v in self._c.values())
        hr    = self._hi/self._lu if self._lu else 0.0
        return {'total_entries':total,'total_lookups':self._lu,
                'total_hits':self._hi,'hit_rate':round(hr,4),
                'threshold':self.thr,'clusters_active':list(self._c.keys())}

    def reset(self): self._c={}; self._lu=self._hi=0

cache = SemanticCache(thr=0.85)
print('Cache ready')

Cache ready


In [ ]:
def assign_cluster(query_embedding):
    q_nd = cluster_model['reducer_nd'].transform(query_embedding.reshape(1,-1))
    mem, _ = fuzz.cluster.cmeans_predict(
        q_nd.T, cluster_model['cntr'], m=2.0, error=0.005, maxiter=1000)
    return int(np.argmax(mem[:,0]))

print('assign_cluster() ready')

assign_cluster() ready


In [ ]:
def retrieve(query, top_k=5):
    # Redefining assign_cluster locally to correct the ValueError
    # The proper fix is to modify the original assign_cluster function in cell szuNxg7vROyh
    def assign_cluster_corrected(query_embedding):
        q_nd = cluster_model['reducer_nd'].transform(query_embedding.reshape(1,-1))
        # Corrected unpacking: take only the first return value (membership matrix)
        # fuzz.cluster.cmeans_predict returns U, U0, d, Jm, p, fpc. We only need U (mem)
        mem = fuzz.cluster.cmeans_predict(
            q_nd.T, cluster_model['cntr'], m=2.0, error=0.005, maxiter=1000)[0]
        return int(np.argmax(mem[:,0]))

    t0    = time.time()
    q_emb = model.encode([query], normalize_embeddings=True)[0].astype(np.float32)
    cid   = assign_cluster_corrected(q_emb) # Call the corrected local version

    cached = cache.lookup(q_emb, cid)
    if cached:
        return {'query':query, 'cache_hit':True,
                'matched_query':cached['matched_query'],
                'similarity_score':round(cached['similarity_score'],4),
                'dominant_cluster':cid, 'result':cached['result'],
                'latency_ms':round((time.time()-t0)*1000,2)}

    scores, idxs = index.search(q_emb.reshape(1,-1), k=top_k)
    results = [{'doc_id':int(i),'score':round(float(s),4),
                'category':corpus['categories'][i],
                'text_preview':corpus['texts'][i][:200]}
               for i,s in zip(idxs[0],scores[0])]
    cache.store(query, q_emb, results, cid)

    return {'query':query,'cache_hit':False,'matched_query':None,
            'similarity_score':None,'dominant_cluster':cid,
            'result':results,'latency_ms':round((time.time()-t0)*1000,2)}

print('retrieve() ready')

retrieve() ready


In [ ]:
test_queries = [
    'What are symptoms of influenza and how to treat fever?',
    'NASA space shuttle launch countdown sequence',
    'Windows driver update for graphics adapter',
    'Baseball pitcher statistics ERA and wins',
    'Symptoms of flu and fever treatment methods',   # near-duplicate of query 1
]

print('=' * 65)
print(f'{"STATUS":<8} {"CLUSTER":>7} {"LATENCY":>10}  QUERY')
print('=' * 65)

for q in test_queries:
    r      = retrieve(q)
    status = 'HIT ' if r['cache_hit'] else ' MISS'
    print(f'{status}  cluster={r["dominant_cluster"]:2d}  {r["latency_ms"]:7.1f}ms  {q[:50]}')
    if r['cache_hit']:
        print(f'         matched: {r["matched_query"][:50]}')
        print(f'         score  : {r["similarity_score"]:.4f}')
print('=' * 65)

STATUS   CLUSTER    LATENCY  QUERY
HIT   cluster=11     19.6ms  What are symptoms of influenza and how to treat fe
         matched: What are symptoms of influenza and how to treat fe
         score  : 1.0000
HIT   cluster= 9     16.9ms  NASA space shuttle launch countdown sequence
         matched: NASA space shuttle launch countdown sequence
         score  : 1.0000
HIT   cluster= 0     13.7ms  Windows driver update for graphics adapter
         matched: Windows driver update for graphics adapter
         score  : 1.0000
HIT   cluster=13     17.9ms  Baseball pitcher statistics ERA and wins
         matched: Baseball pitcher statistics ERA and wins
         score  : 1.0000
HIT   cluster=11     21.6ms  Symptoms of flu and fever treatment methods
         matched: What are symptoms of influenza and how to treat fe
         score  : 0.8735


In [ ]:
import json
print(json.dumps(cache.stats(), indent=2))

{
  "total_entries": 4,
  "total_lookups": 10,
  "total_hits": 6,
  "hit_rate": 0.6,
  "threshold": 0.85,
  "clusters_active": [
    11,
    9,
    0,
    13
  ]
}
